# 00 Margin Data

Identify the best available margin data from the cleaned LushProtein dataset and assess whether it is good enough for product MBA decisions.

In [1]:
from pathlib import Path
import sys


def find_project_root(start=None):
    start = Path.cwd() if start is None else Path(start).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "EDA" / "outputs").exists() and (candidate / "product_mba").exists():
            return candidate
    raise FileNotFoundError("Could not locate project root containing EDA/outputs and product_mba")

PROJECT_ROOT = find_project_root()
EDA_OUTPUTS = PROJECT_ROOT / "EDA" / "outputs_finals"  # FINALS cohort
MBA_DIR = PROJECT_ROOT / "product_mba"
MBA_OUTPUTS = MBA_DIR / "outputs"
MBA_OUTPUTS.mkdir(exist_ok=True)

print("Project root found")
print("EDA outputs: EDA/outputs_finals (finals cohort)")
print("MBA outputs: product_mba/outputs")


Project root found
EDA outputs: EDA/outputs_finals (finals cohort)
MBA outputs: product_mba/outputs


In [2]:
import re
import numpy as np
import pandas as pd

lines = pd.read_parquet(EDA_OUTPUTS / "lines_sku_analysis.parquet")
orders = pd.read_parquet(EDA_OUTPUTS / "orders.parquet")
customers = pd.read_parquet(EDA_OUTPUTS / "customers.parquet")
products = pd.read_parquet(EDA_OUTPUTS / "products.parquet")

lines["order_id"] = lines["order_id"].astype(str)
orders["order_id"] = orders["order_id"].astype(str)
lines["order_date"] = pd.to_datetime(lines["order_date"], utc=True)
lines["year"] = lines["order_date"].dt.year

print("Loaded tables")
print({
    "lines": lines.shape,
    "orders": orders.shape,
    "customers": customers.shape,
    "products": products.shape,
})

def normalize_sku(value):
    if pd.isna(value):
        return pd.NA
    text = str(value).strip().strip("'").strip('"').strip()
    if not text or text.lower() in {"nan", "none", "null"}:
        return pd.NA
    if re.fullmatch(r"\d+\.0", text):
        text = text[:-2]
    return text.upper()

lines["sku_norm"] = lines["Line: SKU"].map(normalize_sku)
products["variant_sku_norm"] = products["Variant SKU"].map(normalize_sku)
products["barcode_norm"] = products["Variant Barcode"].map(normalize_sku)
products["unit_cost"] = pd.to_numeric(products["Cost per item"], errors="coerce")
products["variant_price"] = pd.to_numeric(products["Variant Price"], errors="coerce")

print("Line SKU non-null rate:", f"{lines['sku_norm'].notna().mean():.1%}")
print("Product master cost rows:", int(products["unit_cost"].notna().sum()), "of", len(products))


Loaded tables
{'lines': (14448, 15), 'orders': (9852, 34), 'customers': (6353, 32), 'products': (167, 25)}
Line SKU non-null rate: 83.5%
Product master cost rows: 58 of 167


In [3]:
def make_master_key_map(products, key_col, match_type):
    cols = [key_col, "Handle", "Title", "Variant SKU", "Variant Barcode", "unit_cost", "variant_price", "Status"]
    key_rows = products[cols].dropna(subset=[key_col]).copy()
    key_rows = key_rows.rename(columns={key_col: "match_key"})
    grouped = (
        key_rows.groupby("match_key", dropna=False)
        .agg(
            master_rows=("match_key", "size"),
            distinct_costs=("unit_cost", lambda s: s.dropna().nunique()),
            unit_cost=("unit_cost", "first"),
            variant_price=("variant_price", "first"),
            product_handle=("Handle", "first"),
            product_title=("Title", "first"),
            variant_sku=("Variant SKU", "first"),
            variant_barcode=("Variant Barcode", "first"),
            product_status=("Status", "first"),
        )
        .reset_index()
    )
    grouped["match_type"] = match_type
    grouped["high_confidence_cost"] = (
        grouped["unit_cost"].notna()
        & (grouped["master_rows"] == 1)
        & (grouped["distinct_costs"] <= 1)
    )
    return grouped

sku_map = make_master_key_map(products, "variant_sku_norm", "Variant SKU")
barcode_map = make_master_key_map(products, "barcode_norm", "Variant Barcode")

line_base = lines.copy()
line_base["net_revenue"] = pd.to_numeric(line_base["Line: Total"], errors="coerce").fillna(0)
line_base["quantity"] = pd.to_numeric(line_base["Line: Quantity"], errors="coerce").fillna(0)

sku_join = line_base.merge(sku_map.add_prefix("sku_"), left_on="sku_norm", right_on="sku_match_key", how="left")
joined = sku_join.merge(barcode_map.add_prefix("bar_"), left_on="sku_norm", right_on="bar_match_key", how="left")

use_sku = joined["sku_high_confidence_cost"].fillna(False).astype(bool)
use_barcode = (~use_sku) & joined["bar_high_confidence_cost"].fillna(False).astype(bool)

joined["unit_cost"] = np.select([use_sku, use_barcode], [joined["sku_unit_cost"], joined["bar_unit_cost"]], default=np.nan)
joined["match_type"] = np.select([use_sku, use_barcode], ["Variant SKU", "Variant Barcode"], default=pd.NA)
joined["matched_product_handle"] = np.select([use_sku, use_barcode], [joined["sku_product_handle"], joined["bar_product_handle"]], default=pd.NA)
joined["matched_product_title"] = np.select([use_sku, use_barcode], [joined["sku_product_title"], joined["bar_product_title"]], default=pd.NA)
joined["product_status"] = np.select([use_sku, use_barcode], [joined["sku_product_status"], joined["bar_product_status"]], default=pd.NA)
joined["has_high_confidence_cost"] = joined["unit_cost"].notna()
joined["estimated_cogs"] = np.where(joined["has_high_confidence_cost"], joined["unit_cost"] * joined["quantity"], np.nan)
joined["estimated_gross_profit"] = joined["net_revenue"] - joined["estimated_cogs"]
joined["estimated_margin_pct"] = np.where(joined["net_revenue"] > 0, joined["estimated_gross_profit"] / joined["net_revenue"], np.nan)

margin_cols = [
    "order_id", "customer_id", "order_date", "year", "store", "product_category",
    "Line: Product Handle", "Line: Title", "Line: Variant Title", "Line: SKU", "sku_norm",
    "quantity", "net_revenue", "unit_cost", "estimated_cogs", "estimated_gross_profit",
    "estimated_margin_pct", "has_high_confidence_cost", "match_type",
    "matched_product_handle", "matched_product_title", "product_status",
]
margin_lines = joined[margin_cols].copy()
margin_lines.to_parquet(MBA_OUTPUTS / "margin_line_enriched.parquet", index=False)

print("Saved margin_line_enriched.parquet")
print("High-confidence cost line coverage:", f"{margin_lines['has_high_confidence_cost'].mean():.1%}")
print("High-confidence cost revenue coverage:", f"{margin_lines.loc[margin_lines['has_high_confidence_cost'], 'net_revenue'].sum() / margin_lines['net_revenue'].sum():.1%}")


Saved margin_line_enriched.parquet
High-confidence cost line coverage: 67.1%
High-confidence cost revenue coverage: 58.6%


In [4]:
def coverage_summary(df, group_cols):
    out = (
        df.groupby(group_cols, dropna=False)
        .agg(
            line_items=("order_id", "count"),
            orders=("order_id", "nunique"),
            revenue_sgd=("net_revenue", "sum"),
            covered_revenue_sgd=("net_revenue", lambda s: s[df.loc[s.index, "has_high_confidence_cost"]].sum()),
            covered_lines=("has_high_confidence_cost", "sum"),
            estimated_gross_profit_sgd=("estimated_gross_profit", "sum"),
            estimated_cogs_sgd=("estimated_cogs", "sum"),
        )
        .reset_index()
    )
    out["revenue_coverage_pct"] = np.where(out["revenue_sgd"] > 0, out["covered_revenue_sgd"] / out["revenue_sgd"], np.nan)
    out["line_coverage_pct"] = np.where(out["line_items"] > 0, out["covered_lines"] / out["line_items"], np.nan)
    out["estimated_margin_pct"] = np.where(out["covered_revenue_sgd"] > 0, out["estimated_gross_profit_sgd"] / out["covered_revenue_sgd"], np.nan)
    return out.sort_values("revenue_sgd", ascending=False)

sku_coverage = coverage_summary(margin_lines, ["sku_norm", "Line: Title", "Line: Variant Title", "product_category"])
category_coverage = coverage_summary(margin_lines, ["product_category"])
year_coverage = coverage_summary(margin_lines, ["year"])

missing_cost = (
    margin_lines[~margin_lines["has_high_confidence_cost"]]
    .groupby(["sku_norm", "Line: Title", "Line: Variant Title", "product_category"], dropna=False)
    .agg(line_items=("order_id", "count"), orders=("order_id", "nunique"), revenue_sgd=("net_revenue", "sum"), units=("quantity", "sum"))
    .reset_index()
    .sort_values("revenue_sgd", ascending=False)
)

sku_coverage.to_csv(MBA_OUTPUTS / "margin_coverage_by_sku.csv", index=False)
category_coverage.to_csv(MBA_OUTPUTS / "margin_coverage_by_category.csv", index=False)
year_coverage.to_csv(MBA_OUTPUTS / "margin_coverage_by_year.csv", index=False)
missing_cost.head(50).to_csv(MBA_OUTPUTS / "margin_top_missing_cost.csv", index=False)

all_revenue = margin_lines["net_revenue"].sum()
covered_revenue = margin_lines.loc[margin_lines["has_high_confidence_cost"], "net_revenue"].sum()
sku_relevant = margin_lines[margin_lines["sku_norm"].notna()]
relevant_revenue = sku_relevant["net_revenue"].sum()
relevant_covered = sku_relevant.loc[sku_relevant["has_high_confidence_cost"], "net_revenue"].sum()
all_coverage = covered_revenue / all_revenue if all_revenue else np.nan
relevant_coverage = relevant_covered / relevant_revenue if relevant_revenue else np.nan
core = margin_lines[margin_lines["product_category"].isin(["Lean Protein", "Clear Protein", "Collagen Glow"])]
core_coverage = core.loc[core["has_high_confidence_cost"], "net_revenue"].sum() / core["net_revenue"].sum() if core["net_revenue"].sum() else np.nan

top20 = sku_coverage.head(20)
top20_covered_count = int(top20["revenue_coverage_pct"].fillna(0).gt(0).sum())
if relevant_coverage >= 0.80 and top20_covered_count >= 16:
    assessment = "Strong"
elif relevant_coverage >= 0.50 or (core_coverage >= 0.70 and top20_covered_count >= 10):
    assessment = "Usable with caveats"
else:
    assessment = "Not sufficient"

assessment_text = f"""# Margin Data Assessment

Assessment: **{assessment}**

- Total line revenue coverage with high-confidence cost: {all_coverage:.1%}
- Non-null SKU revenue coverage with high-confidence cost: {relevant_coverage:.1%}
- Core hero category revenue coverage (Lean, Clear, Collagen): {core_coverage:.1%}
- Top 20 SKU/title rows with any cost coverage: {top20_covered_count} / 20
- Product master cost rows: {int(products['unit_cost'].notna().sum())} / {len(products)}

Interpretation: Product master `Cost per item` is the best available internal cost source, and joining against both `Variant SKU` and `Variant Barcode` materially improves coverage. However, margin should be treated as an estimate because cost currency/effective date is not explicitly validated here, many historical lines lack usable SKU keys, and some high-revenue missing rows are barcode formatting or legacy product issues.
"""

(MBA_OUTPUTS / "margin_assessment.md").write_text(assessment_text)
print(assessment_text)
print("\nTop missing-cost rows")
display(missing_cost.head(15))
print("\nCategory coverage")
display(category_coverage)


# Margin Data Assessment

Assessment: **Usable with caveats**

- Total line revenue coverage with high-confidence cost: 58.6%
- Non-null SKU revenue coverage with high-confidence cost: 76.8%
- Core hero category revenue coverage (Lean, Clear, Collagen): 86.0%
- Top 20 SKU/title rows with any cost coverage: 15 / 20
- Product master cost rows: 58 / 167

Interpretation: Product master `Cost per item` is the best available internal cost source, and joining against both `Variant SKU` and `Variant Barcode` materially improves coverage. However, margin should be treated as an estimate because cost currency/effective date is not explicitly validated here, many historical lines lack usable SKU keys, and some high-revenue missing rows are barcode formatting or legacy product issues.


Top missing-cost rows


,sku_norm,Line: Title,Line: Variant Title,product_category,line_items,orders,revenue_sgd,units
227,NaN,Collagen Glow - 300g Pack,NaN,Unknown,137,136,16701.685455,275.0
260,NaN,Lushprotein Lean Protein (1kg/25 servings) (No...,NaN,Unknown,40,40,12827.700000,41.0
198,NaN,"Better Whey Protein - 1KG Pack, Chocolate Dino...",NaN,Unknown,125,125,12269.794545,198.0
259,NaN,Lushprotein Collagen Glow 300gm (30 servings) ...,NaN,Unknown,35,35,9326.300000,35.0
271,NaN,"Plant Protein - 480g Pack, Vanilla",NaN,Unknown,64,64,8367.094242,119.0
205,NaN,"Better Whey Protein - 1KG Pack, Vanilla",NaN,Unknown,76,76,7700.329091,220.0
201,NaN,"Better Whey Protein - 1KG Pack, Matchawhey",NaN,Unknown,43,43,6051.514545,70.0
284,NaN,"Soy Protein Isolate - 1KG Pack, Natural (Unfla...",NaN,Unknown,41,41,5587.270000,81.0
267,NaN,"Plant Protein - 480g Pack, Cocoa Dinosaur",NaN,Unknown,35,35,5546.466667,73.0
202,NaN,"Better Whey Protein - 1KG Pack, Natural (Unfla...",NaN,Unknown,38,38,5124.476667,61.0



Category coverage


,product_category,line_items,orders,revenue_sgd,covered_revenue_sgd,covered_lines,estimated_gross_profit_sgd,estimated_cogs_sgd,revenue_coverage_pct,line_coverage_pct,estimated_margin_pct
6,Unknown,3357,2189,212473.207576,5041.208182,522,2693.028182,2348.18,0.023726,0.155496,0.534203
4,Other,3644,3082,188517.847879,121062.873939,2421,83933.923939,37128.95,0.642183,0.664380,0.693309
1,Clear Protein,2657,2130,176994.175152,159600.916970,2536,117771.706970,41829.21,0.901730,0.954460,0.737914
3,Lean Protein,2420,1939,133979.167273,112726.027273,2177,83009.317273,29716.71,0.841370,0.899587,0.736381
2,Collagen Glow,845,839,45716.229394,34541.323030,639,24289.953030,10251.37,0.755559,0.756213,0.703214
5,Soy Protein,448,446,25848.956061,22139.638485,386,19986.998485,2152.64,0.856500,0.861607,0.902770
0,Accessories,1077,1068,10934.647273,10607.271515,1015,8929.311515,1677.96,0.970061,0.942433,0.841810
